# Lab 9: Running a small model locally

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Nov 4, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

DistilGPT-2 is a small language model, about 82 million parameters. It runs
on a laptop CPU, in Colab, or on a GPU. The 8 billion parameter models from
lecture are the same kind of object. This one is small enough to inspect.

You will do five things, and none of them is training:

1. **The next token.** Read the distribution over the vocabulary for one prompt.
2. **Temperature.** Draw from that distribution at two temperatures.
3. **Ingestion and search.** Embed six chunks, store the vectors, and rank
   them against a question.
4. **The orchestrator.** Run a tool only when the text contains the call.
5. **LoRA, as arithmetic.** Count the trained weights in one low-rank update.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **The draws are seeded, so they should match a classmate's.** A mismatch is
  a bug, or a cell that was run out of order. Run All from the top if that
  happens. Sampling is done on CPU on purpose, so a GPU and a laptop agree.
- **The understanding is assessed on Quiz 10**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent will write the softmax in one shot. It is much less likely to tell
you why London outranks Paris here, and Part 6 asks you to.

## Step 0: Setup

The first run downloads two models from Hugging Face. DistilGPT-2 is a few
hundred megabytes. The embedding model is smaller. After that they are cached.

::: {.callout-note}
## A GPU is optional
`cuda` is used when it exists. A Colab GPU, the lab 3080, and a laptop CPU
all finish the notebook. The sampling generator stays on CPU so the seed
does not depend on the device.
:::

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import (AutoModel, AutoModelForCausalLM, AutoTokenizer,
                          logging as tlog)

tlog.set_verbosity_error()

SEED = 6600
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
CAUSAL = "distilgpt2"
EMBED = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(CAUSAL)
causal = AutoModelForCausalLM.from_pretrained(CAUSAL).to(device).eval()

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

n_params = sum(p.numel() for p in causal.parameters())
print(f"torch {torch.__version__}  device {device}")
print(f"{CAUSAL}: {n_params/1e6:.0f}M parameters")

---

# Part 1 · The next token is a distribution

The prompt is fixed. The model returns one score per token in the vocabulary.
The probability of a token is a softmax over those scores. You want the top
of that list, not a sentence.

In [ ]:
PROMPT = "The Eiffel Tower is in the city of"

encoded = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    next_logits = causal(**encoded).logits[0, -1].float().cpu()

print(f"prompt: {PROMPT!r}")
print(f"vocabulary {next_logits.numel():,}  device of the scores: cpu")

## 1a · Top of the distribution

`logits` is a 1D tensor, one score per vocabulary id. Return the `k` highest
tokens as `(text, probability)` pairs, highest first. Decode each id with
the tokenizer and strip the leading space the tokenizer leaves on a word.

In [ ]:
def top_tokens(logits, tokenizer, k=8):
    """1D scores for one position. Return [(token, probability), ...]."""
    # TODO: softmax over the vocabulary, then the k largest probabilities.
    #   tokenizer.decode([i]).strip() is the word to show.
    raise NotImplementedError("top_tokens")


rows = top_tokens(next_logits, tokenizer, k=8)
print(f"{'token':<16} probability")
for token, prob in rows:
    print(f"{token:<16} {prob:.3f}")
print(f"top: {rows[0][0]} {rows[0][1]:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh([t for t, _ in rows][::-1], [p for _, p in rows][::-1], color="#2980b9")
ax.set_xlabel("probability")
ax.set_title(PROMPT)
fig.tight_layout()

London leads, at about 0.11. Paris is on the list and is not first. The chart
in lecture was a larger GPT-2, and Paris led there. This checkpoint is
smaller, and its most likely token is a wrong fact. The distribution is what
the model believes. It is not a lookup of the right city.

---

# Part 2 · Temperature changes the draw

Temperature divides the scores before the softmax. A low temperature makes
the top token nearly certain. A high temperature flattens the distribution,
so unlikely tokens get drawn. One completion is one draw.

## 2a · One draw

`logits` is the same 1D tensor as in Part 1, on CPU. `generator` is a CPU
generator. Divide the scores by `temperature`, turn them into probabilities,
and draw one vocabulary id with `torch.multinomial`. Pass `generator` through,
or the seed will not match.

In [ ]:
def sample_token(logits, temperature, generator):
    """Draw one vocabulary id. logits is 1D, on CPU."""
    # TODO: divide logits by temperature, softmax, then one multinomial draw.
    #   torch.multinomial(..., num_samples=1, generator=generator).item()
    raise NotImplementedError("sample_token")


def draw_many(temperature, n=20):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(SEED)
    ids = [sample_token(next_logits, temperature, generator) for _ in range(n)]
    words = [tokenizer.decode([i]).strip() for i in ids]
    unique = len(set(words))
    print(f"temperature {temperature:<3} unique {unique}")
    print(" ", " | ".join(words))
    return words


low = draw_many(0.5)
high = draw_many(1.5)
print(f"temperature 0.5 unique {len(set(low))}")
print(f"temperature 1.5 unique {len(set(high))}")

At temperature 0.5, twenty draws use 4 different tokens, and London is most
of them. At temperature 1.5, twenty draws use 20 different tokens, and most
of them are not cities. Same prompt, same weights, same seed. The temperature
is why they disagree. A single bad sentence at 1.5 is the draw.

---

# Part 3 · Ingest the chunks, then search

An embedding model turns a piece of text into one vector. Ingestion is
running that model on the documents and storing the vectors. Search embeds
the question later and takes the nearest stored vector. The chat model from
Part 1 is not this model.

In [ ]:
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED)
embed_model = AutoModel.from_pretrained(EMBED).to(device).eval()


def embed(texts):
    """Normalized vectors, one row per text. Mean of the token vectors,
    with padding positions left out."""
    batch = embed_tokenizer(
        list(texts), padding=True, truncation=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        hidden = embed_model(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)
    return F.normalize(summed / counts, dim=-1).cpu()


CHUNKS = [
    "Bring a coat in Paris.",
    "Rain likely in Paris tomorrow.",
    "Umbrella weather in the city.",
    "8°C and wet in Paris.",
    "Pack a jacket for France.",
    "How to train a dog.",
]
QUESTION = "Will it rain in Paris?"

chunk_vecs = embed(CHUNKS)
question_vec = embed([QUESTION])[0]
print(f"{len(CHUNKS)} chunks stored, each a vector of length {chunk_vecs.shape[1]}")

## 3a · Nearest first

`question` has shape `(D,)` and `chunks` has shape `(N, D)`. Both are already
normalized, so the cosine similarity is a dot product. Return the chunk
indices from nearest to farthest.

In [ ]:
def rank_chunks(question, chunks):
    """Indices of chunks, nearest to question first."""
    # TODO: dot question with each row, then sort high to low.
    raise NotImplementedError("rank_chunks")


order = rank_chunks(question_vec, chunk_vecs)
scores = chunk_vecs @ question_vec
print(f"{'cosine':>8}  chunk")
for i in order:
    print(f"{scores[i]:8.3f}  {CHUNKS[i]}")
print(f"nearest: {CHUNKS[order[0]]}")
print(f"farthest: {CHUNKS[order[-1]]}")

"Rain likely in Paris tomorrow." is nearest. "How to train a dog." is last,
and its cosine is about 0. The coat sentence is in the neighborhood and is
not the closest, because the question asked about rain. No weight in either
model was updated. The vectors were stored, then compared.

---

# Part 4 · The orchestrator is listening for one line

The model does not run the tool. It writes text. A program outside the model
reads that text and runs a tool only when the text matches a call. The format
here is a line `tool: weather(City)`. Any other sentence does nothing.

In [ ]:
def weather(city):
    table = {"Paris": "8°C, rain", "London": "12°C, cloudy"}
    return table.get(city, "unknown")


print(weather("Paris"))

## 4a · Match the call, or leave the text alone

Look at each line. If the stripped line starts with `tool: weather(` and ends
with `)`, the city is the text between those. Call `weather` and append a new
line `result: ...` after the original text. If no line matches, return the
text unchanged.

In [ ]:
def apply_tool(text):
    """Run weather when a line is a tool call. Otherwise return text."""
    # TODO: scan the lines. Append one "result: ..." line per matching call.
    #   A line matches when strip() starts with "tool: weather(" and ends with ")".
    raise NotImplementedError("apply_tool")


called = "I should check the forecast.\ntool: weather(Paris)"
plain = "I should check the forecast."
print("with call:")
print(apply_tool(called))
print("---")
print("without call unchanged:", apply_tool(plain) == plain)

The sentence "I should check the forecast." does not start the tool. The
line `tool: weather(Paris)` does, and the result is appended as more text.
That text is what a second call to the model would read.

---

# Part 5 · LoRA, counted

LoRA freezes a weight matrix $W$ of shape $d \times d$ and trains two
smaller matrices. $A$ has shape $r \times d$ and $B$ has shape $d \times r$.
You save $A$ and $B$, not a second copy of $W$.

## 5a · How small is the update

Return three numbers: the count of frozen weights in $W$, the count of
trained weights in $A$ and $B$ together, and the trained count divided by
the frozen count.

In [ ]:
def lora_sizes(d, r):
    """Return frozen count, trained count, and trained/frozen."""
    # TODO: W is d*d. A is r*d. B is d*r.
    raise NotImplementedError("lora_sizes")


frozen, trained, fraction = lora_sizes(4096, 8)
print(f"frozen {frozen:,}")
print(f"trained {trained:,}")
print(f"fraction {fraction * 100:.2f}% of W")

For this one matrix, rank 8 trains 65,536 weights and freezes 16,777,216.
That is 0.39% of $W$. It is not 0.39% of a whole model. A model has many
matrices, and LoRA is usually applied only to some of them.

---

# Part 6 · What this lab showed

Five questions. If you can answer them, you are ready for the Lab 9 part of
Quiz 10.

1. **Why London can outrank Paris** on "the city of" for this checkpoint,
   and why that is still a successful forward pass.
2. **What temperature does to the next-token distribution**, and why twenty
   draws at 1.5 are not twenty bugs.
3. **Why the dog sentence is last** and the rain sentence is first, and why
   those vectors can be stored before the question is written.
4. **Which line runs the tool**, and what happens to a sentence that does not
   match it.
5. **What the 0.39% is a fraction of**: one matrix $W$, not the whole model.

---

# Part 7 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that the bar chart, the two temperature lines, the ranking, and the
   tool result are visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Nov 4, 11:59 PM ET**.

The notebook is graded for completion. Part 6 is the study list for the quiz.

::: {.callout-note}
## Run from the top if a draw looks off
Both temperature lists start from seed 6600. Running the sampling cell twice
is fine, because the generator is reseeded inside the cell. Skipping the cell
that builds `next_logits` is not.
:::